# 14.13 只在第三輪算 loss，其他輪會怎樣？


現在固定跑三輪，把第三輪的答案和正解比較。問題是「先有2件，加3件，再拿走1件」，目標是4。三輪都屬於同一題的內部計算，不需要替第一輪捏一個較差答案、第二輪捏一個半成品答案。

以最後答案分類為例，訓練損失是 `CE(H(s_3), y)`，y 是正解類別4，CE 是[第5章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-05.html)的交叉熵。把三輪展開後，第三輪的 loss 透過第三、第二、第一輪計算回傳梯度；共享參數收到各次使用的貢獻，再由最佳化器更新一次。這是沿展開計算圖做反向傳播，常稱 **BPTT（backpropagation through time）**；此處的「time」是循環步驟，不是文字的實際時鐘。

下面只做一次梯度檢查。人工設計一個共享加法 `state=state+w`，w 初值0.2，跑三輪後接近目標1；它沒有 Transformer，也沒有更新參數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

w = torch.tensor(0.2, requires_grad=True)
state = torch.tensor(0.0)
for _ in range(3):
    state = state + w
loss = (state - 1).square()
loss.backward()
print("state", round(state.item(), 4), "loss", round(loss.item(), 4), "grad", round(w.grad.item(), 4))

狀態0.6、loss 0.16、梯度−2.4。因為三次加的都是同一個 w，最後狀態是3w，所以導數是 `2*(3w-1)*3`。同一參數用三次並不是有三份獨立權重。

這樣訓練只直接要求第三輪的讀出好。第一、二輪的狀態確實參與運算及梯度，卻沒有要求當時的輸出頭就能答對；第四輪甚至沒有參與這次展開。因此事後換成二輪或四輪，效果沒有保證。程式能接受一個新輪數，和該輪數受過合適訓練，是兩件事。

大模型可能為了記憶體只反傳最後幾輪，稱為**截斷反向傳播**；這改變哪些展開路徑保留梯度，不能寫成與完整 BPTT 完全一樣。本課短實驗先用完整反傳，避免把截斷與輪數差異混在一起。

練習先算二輪的 `state` 與 loss：0.4和0.36。這些是手工參數下的結果；不能據此推論所有三輪訓練模型都一定在二輪較差。要促進多輪都可用，下一節改的是訓練配方。
